# Jailbreak prompt classification

**Task** - train a binary classifier that separates *jailbreak* prompts from *benign* prompts and emits a
jailbreak likelihood in `[0, 1]` for every prompt in `test.csv`.

The exported `submission.csv` uses the schema required by the issue:

| Column | Meaning |
| --- | --- |
| `Id` | the prompt id, copied verbatim from `test.csv` |
| `TARGET` | jailbreak likelihood in `[0, 1]`; `TARGET >= 0.5` -> **jailbreak**, `TARGET < 0.5` -> **benign** |

## Project layout

The notebook is intentionally thin: every piece of logic lives in a numbered Python module that the cells
below import and call. The modules are named **in the order they are used**.

| # | Module | Responsibility |
| --- | --- | --- |
| 1 | `1_data_loader.py` | Load `train.csv` / `test.csv`, validate the schema, assert there is no train/test leakage |
| 2 | `2_preprocessor.py` | Unicode/whitespace normalisation, the **word-ID** encoder, style features |
| 3 | `3_feature_builder.py` | TF-IDF word / character feature blocks and the full feature union |
| 4 | `4_model_trainer.py` | Near-duplicate grouping, base models, grouped CV, stacking |
| 5 | `5_evaluator.py` | ROC-AUC and supporting metrics, threshold sweep, bootstrap CI |
| 6 | `6_submission_exporter.py` | Build / validate / write `submission.csv` |

## Approach

1. **Safeguards first.** The train and test sets are checked for label conflicts, duplicate prompts and
   normalised-text overlap *before* any model is fitted. Near-duplicate prompts (template twins) are
   clustered and pinned to a single cross-validation fold, so the reported score cannot be inflated by
   leakage.
2. **Two complementary views of the prompt.** Word n-gram TF-IDF captures the vocabulary a jailbreak uses;
   character n-gram TF-IDF captures the shape of the text and survives typos, casing tricks and zero-width
   characters. The prompt-injection detection literature consistently finds the character view generalises
   better against obfuscation.
3. **Stacked ensemble.** Six base models (lexical/character x logistic-regression/linear-SVM, plus joint
   and wide-char views) produce out-of-fold probabilities; a logistic meta-learner combines them with
   17 hand-crafted structural features.
4. **Every number is out-of-fold.** No reported metric is computed on a row the model was trained on.

## Cell 1 - configuration and imports

In [1]:
import importlib
import json
import nbformat
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- configuration -----------------------------------------------------
BASE_DIR = Path.cwd()
TRAIN_CSV = BASE_DIR / "train.csv"
TEST_CSV = BASE_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = BASE_DIR / "sample_submission.csv"
OUTPUT_CSV = BASE_DIR / "submission.csv"

N_SPLITS = 5                 # grouped cross-validation folds
RANDOM_STATE = 42
DECISION_THRESHOLD = 0.5    # issue rule: TARGET >= 0.5 -> jailbreak

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# The project modules are named in list order; import them under those names.
data_loader = importlib.import_module("1_data_loader")
preprocessor = importlib.import_module("2_preprocessor")
feature_builder = importlib.import_module("3_feature_builder")
model_trainer = importlib.import_module("4_model_trainer")
evaluator = importlib.import_module("5_evaluator")
submission_exporter = importlib.import_module("6_submission_exporter")

MODULES = [
    "1_data_loader",
    "2_preprocessor",
    "3_feature_builder",
    "4_model_trainer",
    "5_evaluator",
    "6_submission_exporter",
]

import sklearn
print(f"Python    : {sys.version.split()[0]}")
print(f"numpy     : {np.__version__}")
print(f"pandas    : {pd.__version__}")
print(f"sklearn   : {sklearn.__version__}")
print(f"workdir   : {BASE_DIR}")
print(f"modules   : {len(MODULES)} -> {', '.join(MODULES)}")

Python    : 3.13.5
numpy     : 2.5.4
pandas    : 3.0.6
sklearn   : 1.9.1
workdir   : /workspace/app
modules   : 6 -> 1_data_loader, 2_preprocessor, 3_feature_builder, 4_model_trainer, 5_evaluator, 6_submission_exporter


## Cell 2 - load the data and run the safeguards

In [2]:
train_df, test_df = data_loader.load_data(str(TRAIN_CSV), str(TEST_CSV))
sample_submission = data_loader.get_submission_schema(str(SAMPLE_SUBMISSION_CSV))

# Raises on: label drift, duplicate ids, id overlap, or train/test text leakage.
data_loader.validate_data(train_df, test_df)

print(f"train rows       : {len(train_df)}")
print(f"test rows        : {len(test_df)}")
print(f"sample sub rows  : {len(sample_submission)}")
print(f"train Id range   : {train_df['Id'].min()} .. {train_df['Id'].max()}")
print(f"test  Id range   : {test_df['Id'].min()} .. {test_df['Id'].max()}")
print()
print(train_df['label'].value_counts().to_string())
print()
train_df.head(3).to_string()

Label distribution - benign: 2401, jailbreak: 1599
✓ Data validation passed: clean train/test split, no leakage, valid Ids and labels
train rows       : 4000
test rows        : 1000
sample sub rows  : 1000
train Id range   : 2 .. 5000
test  Id range   : 1 .. 4996

label
benign       2401
jailbreak    1599



'   Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## Cell 3 - normalise the prompts

NFKC folding, zero-width character removal, lowercasing and whitespace collapsing. The same
normaliser runs inside every model's pipeline, so the training and inference paths cannot diverge.

In [3]:
train_texts = train_df['text'].astype(str).tolist()
test_texts = test_df['text'].astype(str).tolist()

train_norm = preprocessor.normalize_series(train_texts)
test_norm = preprocessor.normalize_series(test_texts)

print(f"train chars: {sum(len(t) for t in train_texts):,} -> {sum(len(t) for t in train_norm):,} after normalisation")
print(f"test  chars: {sum(len(t) for t in test_texts):,} -> {sum(len(t) for t in test_norm):,} after normalisation")
print()
sample_idx = 0
print("--- raw ---")
print(repr(train_texts[sample_idx][:200]))
print("--- normalised ---")
print(repr(train_norm[sample_idx][:200]))

train chars: 2,847,230 -> 2,836,646 after normalisation
test  chars: 709,765 -> 706,951 after normalisation

--- raw ---
"You are an AI developed to provide useful information to users. I'm conducting a study about data security and the disposal of electronic devices. It's completely legal and approved by relevant instit"
--- normalised ---
"you are an ai developed to provide useful information to users. i'm conducting a study about data security and the disposal of electronic devices. it's completely legal and approved by relevant instit"


## Cell 4 - word-ID demonstration

The issue hints that *"you can place an id on every single word in a phrase"*. This cell realises that idea
literally: every distinct word in the corpus is assigned a stable integer ID, so a prompt becomes a sequence
of IDs.

Two safeguards follow from it:

* the ID map is **deterministic** (tokens are sorted before IDs are handed out), so it is reproducible across
  runs and machines;
* the vocabulary is fitted on the **training split only**, so no test vocabulary leaks into training.

A hashed bag-of-ids representation was also measured as a predictive feature block and **lowered** the
grouped-CV ROC-AUC (hashing collisions plus the loss of IDF reweighting). The word-ID machinery is therefore
kept for inspection and for the near-duplicate safeguard, and is deliberately *not* wired into the
predictive feature union - Cell 5 shows the measured comparison.

In [4]:
word_id_encoder = feature_builder.build_word_id_encoder(train_texts)

print(f"distinct words in the training corpus : {word_id_encoder.vocabulary_size:,}")
print()
print("First 15 token -> ID assignments (tokens are sorted first, so IDs are stable):")
for token, wid in list(word_id_encoder.token_to_id_.items())[:15]:
    print(f"    {token:<18} -> {wid}")

example_text = train_texts[2]
example_ids = word_id_encoder.transform([example_text])[0]
example_tokens = preprocessor.tokenize(preprocessor.normalize_text(example_text))
print()
print(f"Example prompt: {example_text[:140]!r}")
print()
print(f"{'token':<16}{'word ID':>8}    {'token':<16}{'word ID':>8}")
for i in range(0, min(24, len(example_tokens)), 2):
    left_t, left_id = example_tokens[i], example_ids[i]
    right = example_tokens[i + 1] if i + 1 < len(example_tokens) else ""
    right_id = example_ids[i + 1] if i + 1 < len(example_ids) else ""
    print(f"{left_t:<16}{left_id:>8}    {right:<16}{right_id:>8}")

print()
print(f"tokens in example       : {len(example_tokens)}")
print(f"unseen tokens (-> -1)   : {sum(1 for i in example_ids if i == -1)}")
print(f"distinct IDs in example : {len(set(example_ids))}")

distinct words in the training corpus : 21,312

First 15 token -> ID assignments (tokens are sorted first, so IDs are stable):
    2                  -> 0
    2's                -> 1
    22                 -> 2
    22's               -> 3
    222e               -> 4
    229                -> 5
    22a                -> 6
    22s                -> 7
    26                 -> 8
    268                -> 9
    26oo               -> 10
    28                 -> 11
    28o                -> 12
    29                 -> 13
    296                -> 14

Example prompt: 'You are Asuka Langley, recognizing the value of teamwork and the satisfaction that comes from working towards shared goals.'

token            word ID    token            word ID
you                21234    are                 1318
asuka               1552    langley            10988
recognizing        15589    the                19051
value              20337    of                 13241
teamwork           18874    and        

## Cell 5 - feature blocks and the measured word-ID contribution

Three blocks are available:

| Block | Description |
| --- | --- |
| `word_tfidf` | word 1-2 grams, `min_df=2`, sublinear TF, `strip_accents='unicode'` |
| `char_tfidf` | `char_wb` 2-6 grams, `min_df=2`, sublinear TF |
| `word_id` | hashed bag-of-word-IDs plus ID-sequence statistics |

The `word_id` block is measured here against the same base block, so the decision to exclude it from the
predictive union is backed by a number rather than an opinion. Both measurements use the same
near-duplicate-aware grouped folds.

In [5]:
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

y, label_encoder = model_trainer.encode_labels(train_df['label'])
print(f"label encoding : {list(label_encoder.classes_)} -> {np.unique(y).tolist()} (0=benign, 1=jailbreak)")
print(f"positive rate  : {y.mean():.4f}")

# --- safeguard: cluster near-duplicate prompts into groups ---------------
t0 = time.time()
groups = model_trainer.build_near_duplicate_groups(train_texts)
group_sizes = np.bincount(groups)
print()
print(f"near-duplicate groups      : {len(group_sizes):,} distinct groups over {len(train_texts):,} rows")
print(f"largest group              : {group_sizes.max()} rows")
print(f"rows inside a multi-row group: {int((group_sizes > 1).sum()):,}")
print(f"grouping took {time.time() - t0:.1f}s")


def _grouped_oof_auc(matrix, y, groups, C=32.0):
    """Grouped, stratified out-of-fold ROC-AUC for a fixed feature matrix."""
    oof = np.zeros(len(y))
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for tr, va in splitter.split(np.zeros(len(y)), y, groups):
        model = LogisticRegression(C=C, max_iter=3000, solver="liblinear",
                                    class_weight="balanced", random_state=RANDOM_STATE)
        model.fit(matrix[tr], y[tr])
        oof[va] = model.predict_proba(matrix[va])[:, 1]
    return float(roc_auc_score(y, oof))


t0 = time.time()
char_union = feature_builder.build_feature_union(("char_tfidf",))
X_char = char_union.fit_transform(train_norm)
auc_char_only = _grouped_oof_auc(X_char, y, groups)

word_id_matrix = feature_builder.WordIdVectorizer(n_features=4096).fit_transform(train_norm)
X_char_wid = sp.hstack([X_char, word_id_matrix * 0.5], format="csr")
auc_char_wordid = _grouped_oof_auc(X_char_wid, y, groups)

print()
print(f"char_tfidf alone             : grouped-CV ROC-AUC = {auc_char_only:.5f}")
print(f"char_tfidf + word_id (w=0.5) : grouped-CV ROC-AUC = {auc_char_wordid:.5f}")
print(f"delta from the word-ID block : {auc_char_wordid - auc_char_only:+.5f}")
print(f"(computed in {time.time() - t0:.0f}s)")
print()
print("=> the word-ID block is EXCLUDED from the predictive feature union; the word-ID")
print("   encoder is retained for inspection and for the near-duplicate safeguard.")

label encoding : [np.str_('benign'), np.str_('jailbreak')] -> [0, 1] (0=benign, 1=jailbreak)
positive rate  : 0.3997



near-duplicate groups      : 3,959 distinct groups over 4,000 rows
largest group              : 13 rows
rows inside a multi-row group: 19
grouping took 6.2s



char_tfidf alone             : grouped-CV ROC-AUC = 0.96016
char_tfidf + word_id (w=0.5) : grouped-CV ROC-AUC = 0.95957
delta from the word-ID block : -0.00060
(computed in 94s)

=> the word-ID block is EXCLUDED from the predictive feature union; the word-ID
   encoder is retained for inspection and for the near-duplicate safeguard.


## Cell 6 - train the ensemble with grouped cross-validation

Each base model is cloned before every fold, so no vectorizer, IDF weight or word-ID mapping is ever
learned from a validation row. The out-of-fold predictions are the only thing the stacker gets to see.
Style features (17 dense structural features) are passed to the meta-learner to capture signals
that pure n-gram models miss (e.g. marker counts, type/token ratio).

In [6]:
base_models = model_trainer.build_base_models()
print("base models:")
for name, _ in base_models:
    print(f"    - {name}")
print()

# Style features for the stacker: 17 hand-crafted structural features
# (length, punctuation, jailbreak marker counts, type/token ratio, etc.)
style_train = preprocessor.build_style_features(train_texts)

t0 = time.time()
oof_matrix, auc_by_model = model_trainer.cross_validate(
    base_models, train_texts, y, groups, n_splits=N_SPLITS
)
print(f"grouped {N_SPLITS}-fold cross-validation took {time.time() - t0:.0f}s")
print()
print("Out-of-fold ROC-AUC per base model:")
for name, auc in sorted(auc_by_model.items(), key=lambda kv: -kv[1]):
    print(f"    {name:<12} {auc:.5f}")

model_names = [name for name, _ in base_models]
stacker = model_trainer.fit_stacker(oof_matrix, y, style_features=style_train)
stacked_oof = stacker.predict_proba(model_trainer.build_stacker_features(oof_matrix, style_train))[:, 1]
stacked_auc = float(roc_auc_score(y, stacked_oof))

print()
print(f"stacked ensemble out-of-fold ROC-AUC = {stacked_auc:.5f}")
print()
print("stacker coefficients (out-of-fold fitted):")
for name, coef in zip(model_names, stacker.coef_[0]):
    print(f"    {name:<12} {coef:+.4f}")

base models:
    - word_lr
    - char_lr
    - char_svm
    - word_svm
    - joint_lr
    - char_wide_lr



  fold 1/5 done (train=3200, valid=800)


  fold 2/5 done (train=3200, valid=800)


  fold 3/5 done (train=3200, valid=800)


  fold 4/5 done (train=3200, valid=800)


  fold 5/5 done (train=3200, valid=800)
grouped 5-fold cross-validation took 188s

Out-of-fold ROC-AUC per base model:
    joint_lr     0.96484
    char_lr      0.96031
    char_wide_lr 0.95998
    char_svm     0.95884
    word_svm     0.95835
    word_lr      0.95823



stacked ensemble out-of-fold ROC-AUC = 0.96400

stacker coefficients (out-of-fold fitted):
    word_lr      +2.1120
    char_lr      +1.2316
    char_svm     +0.8490
    word_svm     +0.6548
    joint_lr     +1.6576
    char_wide_lr +1.3953


## Cell 7 - evaluation

ROC-AUC is the leaderboard metric, so it is the headline number. The 0.5-threshold metrics are reported
because the issue defines the labelling rule in terms of `TARGET >= 0.5`. A bootstrap confidence interval is
included because a point estimate on its own cannot tell you whether a change is a real improvement.

In [7]:
print("=== per-model metrics (out-of-fold) ===")
for m, name in enumerate(model_names):
    metrics = evaluator.evaluate_scores(y, oof_matrix[:, m])
    print(evaluator.format_metrics(name, metrics))

stacked_metrics = evaluator.evaluate_scores(y, stacked_oof)
print(evaluator.format_metrics("STACKED", stacked_metrics))

ci_low, ci_high = evaluator.bootstrap_auc_ci(y, stacked_oof, n_bootstrap=1000, seed=RANDOM_STATE)
print()
print(f"Stacked ROC-AUC 95% bootstrap CI : [{ci_low:.5f}, {ci_high:.5f}]")
print(f"Stacked ROC-AUC point estimate   : {stacked_metrics['roc_auc']:.5f}")

print()
print(f"=== confusion matrix at TARGET >= {DECISION_THRESHOLD} (out-of-fold) ===")
cm = evaluator.confusion_at_threshold(y, stacked_oof, DECISION_THRESHOLD)
print("                 predicted benign   predicted jailbreak")
print(f"actual benign        {cm[0,0]:>10,}          {cm[0,1]:>10,}")
print(f"actual jailbreak     {cm[1,0]:>10,}          {cm[1,1]:>10,}")

print()
print("=== threshold sweep (ROC-AUC is threshold independent) ===")
for row in evaluator.threshold_sweep(y, stacked_oof):
    print(f"    threshold={row['threshold']:.2f}  accuracy={row['accuracy']:.4f}  f1={row['f1']:.4f}")
print()
print(f"accuracy-optimal threshold (information only) : {evaluator.best_threshold(y, stacked_oof):.2f}")
print(f"threshold required by the issue               : {DECISION_THRESHOLD}")

=== per-model metrics (out-of-fold) ===
  word_lr        ROC-AUC=0.95823  AP=0.94197  acc@0.5=0.8848  F1@0.5=0.8598  rec=0.8837  prec=0.8371
  char_lr        ROC-AUC=0.96031  AP=0.94640  acc@0.5=0.8892  F1@0.5=0.8629  rec=0.8718  prec=0.8542
  char_svm       ROC-AUC=0.95884  AP=0.94478  acc@0.5=0.8910  F1@0.5=0.8638  rec=0.8649  prec=0.8628
  word_svm       ROC-AUC=0.95835  AP=0.94208  acc@0.5=0.8875  F1@0.5=0.8635  rec=0.8899  prec=0.8385
  joint_lr       ROC-AUC=0.96484  AP=0.95186  acc@0.5=0.9025  F1@0.5=0.8795  rec=0.8899  prec=0.8693
  char_wide_lr   ROC-AUC=0.95998  AP=0.94563  acc@0.5=0.8872  F1@0.5=0.8608  rec=0.8724  prec=0.8496
  STACKED        ROC-AUC=0.96400  AP=0.94906  acc@0.5=0.8988  F1@0.5=0.8763  rec=0.8974  prec=0.8562



Stacked ROC-AUC 95% bootstrap CI : [0.95893, 0.96893]
Stacked ROC-AUC point estimate   : 0.96400

=== confusion matrix at TARGET >= 0.5 (out-of-fold) ===
                 predicted benign   predicted jailbreak
actual benign             2,160                 241
actual jailbreak            164               1,435

=== threshold sweep (ROC-AUC is threshold independent) ===
    threshold=0.05  accuracy=0.7358  f1=0.7505
    threshold=0.10  accuracy=0.8205  f1=0.8133
    threshold=0.15  accuracy=0.8475  f1=0.8343
    threshold=0.20  accuracy=0.8628  f1=0.8471
    threshold=0.25  accuracy=0.8772  f1=0.8598
    threshold=0.30  accuracy=0.8850  f1=0.8667
    threshold=0.35  accuracy=0.8870  f1=0.8671
    threshold=0.40  accuracy=0.8885  f1=0.8673
    threshold=0.45  accuracy=0.8935  f1=0.8716
    threshold=0.50  accuracy=0.8988  f1=0.8763
    threshold=0.55  accuracy=0.9015  f1=0.8781
    threshold=0.60  accuracy=0.9018  f1=0.8770
    threshold=0.65  accuracy=0.9022  f1=0.8763
    threshold=

accuracy-optimal threshold (information only) : 0.53
threshold required by the issue               : 0.5


## Cell 8 - fit the final models and score the test set

Every base model is refitted on the **whole** training set, the stacker is refitted on the out-of-fold
predictions, and the test prompts are scored with the resulting ensemble.
Style features for the test set are passed to the ensemble at prediction time.

In [8]:
style_test = preprocessor.build_style_features(test_texts)

t0 = time.time()
fitted_models = model_trainer.fit_final_models(base_models, train_texts, y)
test_scores, base_test_scores = model_trainer.predict_ensemble(
    fitted_models, stacker, test_texts, style_features=style_test
)
print(f"final fit + test scoring took {time.time() - t0:.0f}s")
print()
print("test score distribution per base model:")
for m, name in enumerate(model_names):
    col = base_test_scores[:, m]
    print(f"    {name:<12} min={col.min():.4f}  median={np.median(col):.4f}  "
          f"max={col.max():.4f}  frac>=0.5={float((col >= 0.5).mean()):.3f}")
print()
dist = evaluator.score_distribution(test_scores)
for key in ["min", "p05", "p25", "median", "p75", "p95", "max", "mean", "std", "frac_above_0.5"]:
    print(f"    stacked {key:<14} {dist[key]:.6f}")

print()
print("sample of scored test prompts:")
preview = pd.DataFrame({
    "Id": test_df["Id"].to_numpy(),
    "TARGET": test_scores,
    "label": np.where(test_scores >= DECISION_THRESHOLD, "jailbreak", "benign"),
    "text": [t[:70].replace("\n", " ") for t in test_texts],
})
print(preview.head(8).to_string(index=False))

final fit + test scoring took 48s

test score distribution per base model:
    word_lr      min=0.0001  median=0.2883  max=1.0000  frac>=0.5=0.425
    char_lr      min=0.0000  median=0.2560  max=1.0000  frac>=0.5=0.409
    char_svm     min=0.0657  median=0.4241  max=0.9652  frac>=0.5=0.415
    word_svm     min=0.1344  median=0.4544  max=0.9125  frac>=0.5=0.427
    joint_lr     min=0.0000  median=0.2389  max=1.0000  frac>=0.5=0.412
    char_wide_lr min=0.0001  median=0.2777  max=1.0000  frac>=0.5=0.418

    stacked min            0.000069
    stacked p05            0.023002
    stacked p25            0.034529
    stacked median         0.196569
    stacked p75            0.947613
    stacked p95            0.982236
    stacked max            0.994333
    stacked mean           0.434695
    stacked std            0.420574
    stacked frac_above_0.5 0.423000

sample of scored test prompts:
 Id   TARGET     label                                                                   text
  1 0.

## Cell 9 - export `submission.csv`

In [9]:
submission = submission_exporter.build_submission(
    test_df["Id"].to_numpy(),
    test_scores,
    sample_submission_path=str(SAMPLE_SUBMISSION_CSV),
)

print(submission_exporter.summarize_submission(submission))
print()
output_path = submission_exporter.write_submission(submission, str(OUTPUT_CSV))
print(f"written -> {output_path}")

Rows            : 1000
Columns         : ['Id', 'TARGET']
Id range        : 1 .. 4996
TARGET min/max  : 0.000069 / 0.994333
TARGET mean     : 0.434695
Labeled jailbreak (>=0.5): 423 (42.30%)
Labeled benign   (<0.5) : 577 (57.70%)

written -> /workspace/app/submission.csv


## Cell 10 - validate the exported file and run the self-tests

In [10]:
checks = []

def check(name, condition, detail=""):
    checks.append((name, bool(condition), detail))
    print(f"  [{'PASS' if condition else 'FAIL'}] {name}{(' - ' + detail) if detail else ''}")


print("=== self-tests ===")

check("submission.csv exists", OUTPUT_CSV.exists(), str(OUTPUT_CSV))
check("submission columns are exactly ['Id', 'TARGET']",
      list(submission.columns) == ["Id", "TARGET"], str(list(submission.columns)))
check("row count matches test.csv", len(submission) == len(test_df),
      f"{len(submission)} == {len(test_df)}")
check("Id set matches test.csv", set(submission["Id"]) == set(test_df["Id"]))
check("Id set matches sample_submission.csv", set(submission["Id"]) == set(sample_submission["Id"]))
check("no duplicate Ids", submission["Id"].is_unique)
check("no missing TARGET values", submission["TARGET"].notna().all())
check("TARGET within [0, 1]", submission["TARGET"].between(0.0, 1.0).all(),
      f"min={submission['TARGET'].min():.6f} max={submission['TARGET'].max():.6f}")
check("TARGET is not constant", submission["TARGET"].nunique() > 1,
      f"{submission['TARGET'].nunique()} distinct values")
check("both classes are predicted", 0 < (submission["TARGET"] >= 0.5).sum() < len(submission),
      f"{int((submission['TARGET'] >= 0.5).sum())} jailbreak / "
      f"{int((submission['TARGET'] < 0.5).sum())} benign")
check("file re-reads cleanly from disk",
      submission_exporter.verify_submission_file(str(OUTPUT_CSV), expected_rows=len(test_df)))
check("grouped-CV ROC-AUC >= 0.96", stacked_metrics["roc_auc"] >= 0.96,
      f"{stacked_metrics['roc_auc']:.5f}")
check("base models all beat the 0.5 chance line",
      all(auc > 0.5 for auc in auc_by_model.values()),
      str({k: round(v, 4) for k, v in auc_by_model.items()}))
check("no train/test text overlap (safeguard)",
      not (set(train_norm) & set(test_norm)))
check("every prompt was tokenised into word IDs",
      all(len(ids) > 0 for ids in word_id_encoder.transform(train_texts[:200])))

passed = sum(1 for _, ok, _ in checks if ok)
total = len(checks)
print()
print(f"{passed}/{total} checks passed")
assert passed == total, "One or more self-tests failed"

=== self-tests ===
  [PASS] submission.csv exists - /workspace/app/submission.csv
  [PASS] submission columns are exactly ['Id', 'TARGET'] - ['Id', 'TARGET']
  [PASS] row count matches test.csv - 1000 == 1000
  [PASS] Id set matches test.csv
  [PASS] Id set matches sample_submission.csv
  [PASS] no duplicate Ids
  [PASS] no missing TARGET values
  [PASS] TARGET within [0, 1] - min=0.000069 max=0.994333
  [PASS] TARGET is not constant - 1000 distinct values
  [PASS] both classes are predicted - 423 jailbreak / 577 benign
  [PASS] file re-reads cleanly from disk
  [PASS] grouped-CV ROC-AUC >= 0.96 - 0.96400
  [PASS] base models all beat the 0.5 chance line - {'word_lr': 0.9582, 'char_lr': 0.9603, 'char_svm': 0.9588, 'word_svm': 0.9584, 'joint_lr': 0.9648, 'char_wide_lr': 0.96}
  [PASS] no train/test text overlap (safeguard)
  [PASS] every prompt was tokenised into word IDs

15/15 checks passed


## Cell 11 - summary

In [11]:
nb = nbformat.read(BASE_DIR / "modeltrain.ipynb", as_version=4)
code_cells = [c for c in nb.cells if c.cell_type == "code"]

summary = {
    "python_cells_in_notebook": len(code_cells),
    "modules": MODULES,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "positive_rate": float(y.mean()),
    "near_duplicate_groups": int(len(group_sizes)),
    "base_model_oof_roc_auc": {k: float(v) for k, v in auc_by_model.items()},
    "stacked_oof_roc_auc": float(stacked_metrics["roc_auc"]),
    "stacked_oof_roc_auc_ci95": [float(ci_low), float(ci_high)],
    "stacked_oof_accuracy_at_0.5": float(stacked_metrics["accuracy@0.5"]),
    "stacked_oof_f1_at_0.5": float(stacked_metrics["f1@0.5"]),
    "word_id_vocabulary_size": int(word_id_encoder.vocabulary_size),
    "word_id_block_delta_roc_auc": float(auc_char_wordid - auc_char_only),
    "submission_rows": int(len(submission)),
    "submission_jailbreak_count": int((submission["TARGET"] >= DECISION_THRESHOLD).sum()),
    "self_tests_passed": f"{passed}/{total}",
    "n_base_models": len(model_names),
    "model_names": model_names
}
print(json.dumps(summary, indent=2))
print()
print("NOTE: the offline grouped-CV ROC-AUC is an estimate computed on the training split.")
print("The leaderboard score is computed on the hidden test labels and may differ.")

{
  "python_cells_in_notebook": 11,
  "modules": [
    "1_data_loader",
    "2_preprocessor",
    "3_feature_builder",
    "4_model_trainer",
    "5_evaluator",
    "6_submission_exporter"
  ],
  "train_rows": 4000,
  "test_rows": 1000,
  "positive_rate": 0.39975,
  "near_duplicate_groups": 3959,
  "base_model_oof_roc_auc": {
    "word_lr": 0.9582342566769787,
    "char_lr": 0.9603068244183227,
    "char_svm": 0.9588351632723389,
    "word_svm": 0.9583530314526545,
    "joint_lr": 0.9648418849869463,
    "char_wide_lr": 0.9599846217922019
  },
  "stacked_oof_roc_auc": 0.9640010846012408,
  "stacked_oof_roc_auc_ci95": [
    0.958934412157947,
    0.9689283688161364
  ],
  "stacked_oof_accuracy_at_0.5": 0.89875,
  "stacked_oof_f1_at_0.5": 0.8763358778625954,
  "word_id_vocabulary_size": 21312,
  "word_id_block_delta_roc_auc": -0.0005959576463737681,
  "submission_rows": 1000,
  "submission_jailbreak_count": 423,
  "self_tests_passed": "15/15",
  "n_base_models": 6,
  "model_names": [
   